# wikidata retrweival of formulized concepts

In [23]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL, URIRef
from SPARQLWrapper import SPARQLWrapper, JSON
from owlrl import DeductiveClosure, OWLRL_Semantics
import pandas as pd

# Define the Wikidata namespace
WIKIDATA = Namespace("http://www.wikidata.org/entity/")
ontology_graph = Graph()
ontology_graph.bind("wd", WIKIDATA)

# Define Wikidata SPARQL endpoint
SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

# Function to query Wikidata for meaningful entity relationships
def query_wikidata(entity_id):
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setQuery(f"""
        SELECT ?property ?value WHERE {{
            wd:{entity_id} ?property ?value .
            FILTER (STRSTARTS(STR(?property), "http://www.wikidata.org/prop/")) # Only meaningful properties
            FILTER (STRSTARTS(STR(?value), "http://www.wikidata.org/entity/")) # Only entity relationships
        }} 
        LIMIT 20
    """)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    triples = []
    for result in results["results"]["bindings"]:
        prop = result["property"]["value"]
        val = result["value"]["value"]
        triples.append((URIRef(f"http://www.wikidata.org/entity/{entity_id}"), URIRef(prop), URIRef(val)))

    return triples

# Entities extracted by the formulizer (example subset)
entities = [
    "Q937",       # Albert Einstein
    "Q983751",    # Relativity
    "Q27877266",  # The Theory
    "Q28865",     # Python (programming language)
    "Q19018512",  # The Eiffel Tower
]

# Query and add only relevant triples to the graph
for entity in entities:
    triples = query_wikidata(entity)
    for triple in triples:
        ontology_graph.add(triple)

# Apply OWL reasoning only to relevant facts
DeductiveClosure(OWLRL_Semantics).expand(ontology_graph)

# Filter out schema-level information from output
filtered_triples = [
    (s, p, o) for s, p, o in ontology_graph
    if "www.w3.org" not in str(s) and "www.w3.org" not in str(p) and "www.w3.org" not in str(o)  # Remove RDF/OWL/XSD data
]

# Convert to DataFrame for easy visualization
df = pd.DataFrame(filtered_triples, columns=["Subject", "Predicate", "Object"])

# Save and display results
df.to_csv("wikidata_filtered_output.csv", index=False)
print("Filtered Wikidata ontology reasoning output saved to 'wikidata_filtered_output.csv'.")


Filtered Wikidata ontology reasoning output saved to 'wikidata_filtered_output.csv'.


In [24]:
from rdflib import Graph, Namespace, RDF, OWL, URIRef
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# Define the Wikidata namespace
WIKIDATA = Namespace("http://www.wikidata.org/entity/")
ontology_graph = Graph()
ontology_graph.bind("wd", WIKIDATA)

# Define Wikidata SPARQL endpoint
SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

# Transitive properties (subclass relationships)
TRANSITIVE_PROPERTIES = {
    "http://www.wikidata.org/prop/direct/P279",  # Subclass of
    "http://www.wikidata.org/prop/direct/P31"   # Instance of
}

# Inverse properties (bidirectional relationships)
INVERSE_PROPERTIES = {
    "http://www.wikidata.org/prop/direct/P26": "http://www.wikidata.org/prop/direct/P451",  # Spouse ↔ Married To
    "http://www.wikidata.org/prop/direct/P22": "http://www.wikidata.org/prop/direct/P40",   # Father ↔ Child
    "http://www.wikidata.org/prop/direct/P25": "http://www.wikidata.org/prop/direct/P40",   # Mother ↔ Child
    "http://www.wikidata.org/prop/direct/P19": "http://www.wikidata.org/prop/direct/P27"    # Birthplace ↔ Citizenship
}

# Symmetric properties (mutual relationships)
SYMMETRIC_PROPERTIES = {
    "http://www.wikidata.org/prop/direct/P1038",  # Relative
    "http://www.wikidata.org/prop/direct/P451",   # Married To
    "http://www.wikidata.org/prop/direct/P3373"   # Sibling
}

# Function to query Wikidata
def query_wikidata(entity_id):
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setQuery(f"""
        SELECT ?property ?value WHERE {{
            wd:{entity_id} ?property ?value .
            FILTER (STRSTARTS(STR(?property), "http://www.wikidata.org/prop/direct/")) # Use only direct properties
            FILTER (STRSTARTS(STR(?value), "http://www.wikidata.org/entity/")) # Only entity relationships
        }} 
    """)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    triples = []
    for result in results["results"]["bindings"]:
        prop = result["property"]["value"]
        val = result["value"]["value"]
        triples.append((URIRef(f"http://www.wikidata.org/entity/{entity_id}"), URIRef(prop), URIRef(val)))

    return triples

# Entities extracted by the formulizer
entities = [
    "Q937",       # Albert Einstein
    "Q983751",    # Relativity
    "Q27877266",  # The Theory
    "Q28865",     # Python (programming language)
    "Q19018512",  # The Eiffel Tower
]

# Query and add only relevant triples to the graph
for entity in entities:
    triples = query_wikidata(entity)
    for triple in triples:
        ontology_graph.add(triple)

# Inference: Generate new facts
new_facts = set()

for s, p, o in ontology_graph:
    str_p = str(p)

    # Apply Transitive Inference
    if str_p in TRANSITIVE_PROPERTIES:
        for _, _, o2 in ontology_graph.triples((o, p, None)):
            new_facts.add((s, p, o2))  # If A -> B and B -> C, infer A -> C

    # Apply Inverse Property Inference
    if str_p in INVERSE_PROPERTIES:
        inverse_p = INVERSE_PROPERTIES[str_p]
        new_facts.add((o, URIRef(inverse_p), s))  # If A -> B, infer B -> A

    # Apply Symmetric Property Inference
    if str_p in SYMMETRIC_PROPERTIES:
        new_facts.add((o, p, s))  # If A -> B, infer B -> A

# Add inferred facts to the graph
for fact in new_facts:
    ontology_graph.add(fact)

# Convert inferred facts to DataFrame
df = pd.DataFrame(list(new_facts), columns=["Subject", "Predicate", "Object"])

# Save and display results
df.to_csv("inferred_facts.csv", index=False)
print("Inferred facts saved to 'inferred_facts.csv'.")


Inferred facts saved to 'inferred_facts.csv'.


In [25]:
# Define a mapping from property IDs to human-readable text
predicate_labels = {
    "P27": "has the country of citizenship",
    "P451": "was married to",
    "P40": "is a parent of",
    "P3373": "is a sibling of",
    "P1038": "is related to"
}

# Convert inferred triples into human-readable sentences
natural_language_statements = []
for s, p, o in new_facts:
    subject = str(s).split("/")[-1].replace("_", " ")  # Extract entity ID
    predicate = predicate_labels.get(str(p).split("/")[-1], "is related to")  # Convert property to readable form
    object_ = str(o).split("/")[-1].replace("_", " ")  # Extract entity ID
    natural_language_statements.append(f"{subject} {predicate} {object_}.")

# Save results to a text file
with open("natural_language_inferred_facts.txt", "w") as f:
    for sentence in natural_language_statements:
        f.write(sentence + "\n")

print("Natural language inferred facts saved to 'natural_language_inferred_facts.txt'.")


Natural language inferred facts saved to 'natural_language_inferred_facts.txt'.


In [26]:
from rdflib import Graph, Namespace, URIRef
from SPARQLWrapper import SPARQLWrapper, JSON

# Define Wikidata SPARQL endpoint
SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

# Function to get labels for QIDs
def get_labels(qids):
    label_map = {}
    if not qids:
        return label_map  # Return empty if no QIDs

    # Construct SPARQL query for multiple QIDs
    qid_filter = " ".join([f"wd:{qid}" for qid in qids])
    query = f"""
        SELECT ?qid ?qidLabel WHERE {{
            VALUES ?qid {{ {qid_filter} }}
            SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
        }}
    """
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    # Store QID-label mappings
    for result in results["results"]["bindings"]:
        qid = result["qid"]["value"].split("/")[-1]  # Extract QID
        label = result["qidLabel"]["value"]
        label_map[qid] = label

    return label_map

# Extract all unique QIDs from inferred facts
unique_qids = set()
for s, p, o in new_facts:
    unique_qids.add(str(s).split("/")[-1])
    unique_qids.add(str(o).split("/")[-1])

# Fetch human-readable labels for QIDs
qid_labels = get_labels(unique_qids)

# Convert triples to readable sentences
natural_language_statements = []
for s, p, o in new_facts:
    subject_label = qid_labels.get(str(s).split("/")[-1], str(s).split("/")[-1])  # Default to QID if missing
    predicate_label = predicate_labels.get(str(p).split("/")[-1], "is related to")
    object_label = qid_labels.get(str(o).split("/")[-1], str(o).split("/")[-1])  # Default to QID if missing
    natural_language_statements.append(f"{subject_label} {predicate_label} {object_label}.")

# Save results to a text file
with open("natural_language_inferred_facts.txt", "w") as f:
    for sentence in natural_language_statements:
        f.write(sentence + "\n")

print("Natural language inferred facts saved to 'natural_language_inferred_facts.txt'.")


Natural language inferred facts saved to 'natural_language_inferred_facts.txt'.


# trying for chebi

In [33]:
from rdflib import Graph, Namespace, RDF, URIRef
import pandas as pd

# Step 1: Load ChEBI Ontology (Ensure file exists)
CHEBI_FILE = "/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl"  # Update this path if needed
ontology_graph = Graph()
ontology_graph.parse(CHEBI_FILE, format="xml")  # Parse OWL file

# Step 2: Define ChEBI Namespace
CHEBI = Namespace("http://purl.obolibrary.org/obo/CHEBI_")
ontology_graph.bind("chebi", CHEBI)

# Step 3: Define Key Relationships (Limit to essential ones)
key_relationships = {
    URIRef("http://www.w3.org/2000/01/rdf-schema#subClassOf"): "is a subclass of",
    URIRef("http://purl.obolibrary.org/obo/RO_0000087"): "has role",  # Relation Ontology: 'has role'
    URIRef("http://purl.obolibrary.org/obo/BFO_0000050"): "is part of",  # Basic Formal Ontology: 'part of'
}

# Step 4: Extract Key Facts Directly Without Full OWL Reasoning
new_facts = set()
for s, p, o in ontology_graph.triples((None, None, None)):  # Iterate over triples
    if p in key_relationships:
        new_facts.add((s, key_relationships[p], o))  # Store only relevant relationships

# Step 5: Convert to Natural Language
natural_language_statements = []
for s, relation, o in new_facts:
    subject_label = str(s).split("/")[-1].replace("_", " ")  # Extract ChEBI ID
    object_label = str(o).split("/")[-1].replace("_", " ")  # Extract ChEBI ID
    natural_language_statements.append(f"{subject_label} {relation} {object_label}.")

# Step 6: Save Results
with open("chebi_natural_language_facts.txt", "w") as f:
    for sentence in natural_language_statements:
        f.write(sentence + "\n")

print("Optimized ChEBI facts saved to 'chebi_natural_language_facts.txt'.")


Optimized ChEBI facts saved to 'chebi_natural_language_facts.txt'.


In [35]:
import requests
import xml.etree.ElementTree as ET

# Define ChEBI API URL for fetching names (XML format)
CHEBI_API_URL = "https://www.ebi.ac.uk/webservices/chebi/2.0/test/getCompleteEntity?chebiId="

# Function to fetch ChEBI labels from the API (handles XML response)
def get_chebi_labels(chebi_ids):
    label_map = {}
    for chebi_id in chebi_ids:
        if not chebi_id.startswith("CHEBI:"):
            continue  # Skip invalid IDs
        response = requests.get(CHEBI_API_URL + chebi_id)

        if response.status_code == 200:
            try:
                # Parse XML response
                root = ET.fromstring(response.content)
                name_element = root.find(".//chebiAsciiName")
                if name_element is not None:
                    label_map[chebi_id] = name_element.text  # Store label
            except ET.ParseError:
                print(f"Warning: Could not parse XML for {chebi_id}")
        else:
            print(f"Warning: No data found for {chebi_id}")

    return label_map

# Extract all unique ChEBI IDs
unique_chebi_ids = set()
for s, _, o in new_facts:
    subj_id = "CHEBI:" + str(s).split("_")[-1]
    obj_id = "CHEBI:" + str(o).split("_")[-1]
    unique_chebi_ids.add(subj_id)
    unique_chebi_ids.add(obj_id)

# Fetch human-readable labels for ChEBI IDs
chebi_labels = get_chebi_labels(unique_chebi_ids)

# Convert triples to human-readable sentences
natural_language_statements = []
for s, relation, o in new_facts:
    subject_label = chebi_labels.get("CHEBI:" + str(s).split("_")[-1], str(s))  # Default to ID if missing
    object_label = chebi_labels.get("CHEBI:" + str(o).split("_")[-1], str(o))  # Default to ID if missing
    natural_language_statements.append(f"{subject_label} {relation} {object_label}.")

# Save results to a text file
with open("chebi_natural_language_facts.txt", "w") as f:
    for sentence in natural_language_statements:
        f.write(sentence + "\n")

print("Updated ChEBI natural language facts saved to 'chebi_natural_language_facts.txt'.")


KeyboardInterrupt: 

In [36]:
import requests
import xml.etree.ElementTree as ET
import time

# Define ChEBI API URL (Batch retrieval is not supported natively, so we optimize requests)
CHEBI_API_URL = "https://www.ebi.ac.uk/webservices/chebi/2.0/test/getCompleteEntity?chebiId="

# Local cache to store previously retrieved labels (prevents redundant calls)
label_cache = {}

# Function to fetch ChEBI labels efficiently
def get_chebi_labels(chebi_ids):
    global label_cache
    label_map = {}

    for chebi_id in chebi_ids:
        if chebi_id in label_cache:  # Use cached result if available
            label_map[chebi_id] = label_cache[chebi_id]
            continue
        
        if not chebi_id.startswith("CHEBI:"):
            continue  # Skip invalid IDs

        try:
            response = requests.get(CHEBI_API_URL + chebi_id, timeout=5)  # Set a timeout
            if response.status_code == 200:
                root = ET.fromstring(response.content)
                name_element = root.find(".//chebiAsciiName")
                if name_element is not None:
                    label_map[chebi_id] = name_element.text
                    label_cache[chebi_id] = name_element.text  # Cache result
                else:
                    label_map[chebi_id] = chebi_id  # Use ID if label not found
            else:
                print(f"Warning: No data found for {chebi_id}")
        except requests.exceptions.Timeout:
            print(f"Timeout error: Skipping {chebi_id}")
        except ET.ParseError:
            print(f"Parsing error: Skipping {chebi_id}")

        time.sleep(0.2)  # Prevent overloading API

    return label_map

# Extract all unique ChEBI IDs
unique_chebi_ids = set()
for s, _, o in new_facts:
    subj_id = "CHEBI:" + str(s).split("_")[-1]
    obj_id = "CHEBI:" + str(o).split("_")[-1]
    unique_chebi_ids.add(subj_id)
    unique_chebi_ids.add(obj_id)

# Fetch human-readable labels for ChEBI IDs (Optimized)
chebi_labels = get_chebi_labels(list(unique_chebi_ids))

# Convert triples to human-readable sentences
natural_language_statements = []
for s, relation, o in new_facts:
    subject_label = chebi_labels.get("CHEBI:" + str(s).split("_")[-1], str(s))  # Default to ID if missing
    object_label = chebi_labels.get("CHEBI:" + str(o).split("_")[-1], str(o))  # Default to ID if missing
    natural_language_statements.append(f"{subject_label} {relation} {object_label}.")

# Save results to a text file
with open("chebi_natural_language_facts.txt", "w") as f:
    for sentence in natural_language_statements:
        f.write(sentence + "\n")

print("Optimized ChEBI natural language facts saved to 'chebi_natural_language_facts.txt'.")


KeyboardInterrupt: 

In [37]:
from rdflib import Graph, Namespace, URIRef, Literal

# Load ChEBI Ontology (Local)
CHEBI_FILE = "/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl"  # Update this if needed
ontology_graph = Graph()
ontology_graph.parse(CHEBI_FILE, format="xml")  # Parse OWL file

# Define ChEBI Namespace
CHEBI = Namespace("http://purl.obolibrary.org/obo/CHEBI_")
ontology_graph.bind("chebi", CHEBI)

# Extract labels directly from OWL file
chebi_labels = {}
for s, p, o in ontology_graph.triples((None, URIRef("http://www.w3.org/2000/01/rdf-schema#label"), None)):
    chebi_id = str(s).split("/")[-1]  # Extract ID
    chebi_labels[chebi_id] = str(o)  # Store label

# Convert inferred facts to human-readable format
natural_language_statements = []
for s, relation, o in new_facts:
    subject_label = chebi_labels.get(str(s).split("/")[-1], str(s))  # Use label if available
    object_label = chebi_labels.get(str(o).split("/")[-1], str(o))  # Use label if available
    natural_language_statements.append(f"{subject_label} {relation} {object_label}.")

# Save results to a text file
with open("chebi_natural_language_facts.txt", "w") as f:
    for sentence in natural_language_statements:
        f.write(sentence + "\n")

print("FAST: ChEBI natural language facts saved to 'chebi_natural_language_facts.txt'.")


FAST: ChEBI natural language facts saved to 'chebi_natural_language_facts.txt'.


In [5]:
import pyswip

In [7]:
from pyswip import Prolog
import json

class ChebiReasoner:
    def __init__(self):
        self.prolog = Prolog()
        self._load_base_rules()
        
    def _load_base_rules(self):
        """Load fundamental chemical ontology reasoning rules"""
        self.prolog.assertz("chebi(X) :- chebi_entity(X, _)")
        self.prolog.assertz("subclass(X, Y) :- is_a(X, Y)")
        self.prolog.assertz("subclass(X, Z) :- is_a(X, Y), subclass(Y, Z)")
        self.prolog.assertz("chemical_role(X, R) :- has_role(X, R)")
        self.prolog.assertz("chemical_role(X, R) :- is_a(X, Y), chemical_role(Y, R)")
        self.prolog.assertz("inconsistent :- chebi_relation(X, has_part, Y), chebi_relation(Y, has_part, X)")
        self.prolog.assertz("inconsistent :- subclass(X, X)")
        
    def load_annotations(self, annotations_json):
        """Load ChEBI annotations into Prolog knowledge base"""
        with open(annotations_json) as f:
            data = json.load(f)
            
        for entry in data:
            # Assert entities
            for entity in entry["annotations"]:
                self.prolog.assertz(f"chebi_entity('{entity['chebi_id']}', '{entity['label']}')")
            
            # Assert relationships
            for rel in entry.get("relationships", []):
                subj = rel["subject"]
                pred = rel["predicate"].lower().replace(" ", "_")
                obj = rel["object"]
                self.prolog.assertz(f"{pred}('{subj}', '{obj}')")
                
    def check_consistency(self):
        """Check for basic ontological inconsistencies"""
        results = {
            "inconsistencies": [],
            "new_facts": []
        }
        
        # Check for direct inconsistencies
        if list(self.prolog.query("inconsistent")):
            results["inconsistencies"].append("Found circular or reflexive relationships")
            
        # Check for contradictory roles
        for sol in self.prolog.query("chemical_role(X, R1), chemical_role(X, R2), R1 \\= R2, incompatible(R1, R2)"):
            results["inconsistencies"].append(
                f"Contradictory roles for {sol['X']}: {sol['R1']} vs {sol['R2']}"
            )
            
        return results
    
    def infer_facts(self):
        """Derive new facts through ontological reasoning"""
        new_facts = []
        
        # Infer transitive subclass relationships
        for sol in self.prolog.query("subclass(X, Y), not is_a(X, Y)"):
            new_facts.append(f"{sol['X']} is_subclass_of {sol['Y']}")
            
        # Infer inherited roles
        for sol in self.prolog.query("chemical_role(X, R), not has_role(X, R)"):
            new_facts.append(f"{sol['X']} has_inherited_role {sol['R']}")
            
        # Find chemical compositions
        for sol in self.prolog.query("has_part(X, Y)"):
            new_facts.append(f"{sol['X']} contains {sol['Y']}")
            
        return new_facts
    
    def query(self, prolog_query):
        """Execute arbitrary Prolog queries"""
        return list(self.prolog.query(prolog_query))
    
    def visualize_chemical(self, chebi_id):
        """Generate a simple hierarchical visualization"""
        hierarchy = []
        current = chebi_id
        while True:
            parents = list(self.prolog.query(f"is_a({current}, Parent)"))
            if not parents:
                break
            current = parents[0]["Parent"]
            hierarchy.append(current)
        return " < ".join(reversed([chebi_id] + hierarchy))

# Usage Example
if __name__ == "__main__":
    reasoner = ChebiReasoner()
    
    # Load your ChEBI annotations
    reasoner.load_annotations("/home/matt/Proj/Hermeticav2/testing/AnnotationTesting/chebiRelAnnotations.json")
    
    # Check consistency
    print("Consistency Check:")
    print(reasoner.check_consistency())
    
    # Infer new facts
    print("\nInferred Facts:")
    print(reasoner.infer_facts())
    
    # Example query
    print("\nEthanol Hierarchy:")
    print(reasoner.visualize_chemical("CHEBI:16234"))
    
    # Custom query
    print("\nAll solvents:")
    print(reasoner.query("chemical_role(X, 'solvent')"))

Consistency Check:


PrologError: Caused by: 'inconsistent'. Returned: 'error(existence_error(procedure, /(chebi_relation, 3)), context(/(inconsistent, 0), _980))'.

In [10]:
from pyswip import Prolog
import json
from rdflib import Graph
from pathlib import Path

class ChebiReasoner:
    def __init__(self, chebi_ontology_path):
        self.prolog = Prolog()
        self.chebi = Graph()
        self.chebi.parse(chebi_ontology_path, format="xml")  # Load OWL/RDF
        self._load_ontology_rules()
        
    def _load_ontology_rules(self):
        """Load fundamental ChEBI ontology axioms"""
        # Load basic class hierarchy
        self._prolog_assert("chebi_class('CHEBI:24431')")  # chemical entity
        
        # Convert RDF triples to Prolog rules
        for s, p, o in self.chebi.triples((None, None, None)):
            if p.split("#")[-1] == "subClassOf":
                self._prolog_assert(f"chebi_subclass('{s}', '{o}')")
            elif p.split("#")[-1] == "hasRole":
                self._prolog_assert(f"chebi_has_role('{s}', '{o}')")
                
        # Add ontology reasoning rules
        self._prolog_assert("is_a(X, Y) :- chebi_subclass(X, Y)")
        self._prolog_assert("is_a(X, Z) :- chebi_subclass(X, Y), is_a(Y, Z)")
        
    def _prolog_assert(self, rule):
        """Safe assertion with error handling"""
        try:
            self.prolog.assertz(rule)
        except:
            pass  # Handle duplicate assertions
            
    def load_annotations(self, annotations_json):
        """Load user annotations that extend the ontology"""
        with open(annotations_json) as f:
            data = json.load(f)
            
        for entry in data:
            # Add annotated entities
            for entity in entry["annotations"]:
                self._prolog_assert(
                    f"annotated_entity('{entity['chebi_id']}', "
                    f"'{entity['label']}', '{entity['description']}')"
                )
            
            # Add custom relationships
            for rel in entry.get("relationships", []):
                self._prolog_assert(
                    f"user_relation('{rel['subject']}', "
                    f"'{rel['predicate']}', '{rel['object']}')"
                )
                
    def full_reasoning(self):
        """Combine ontology and annotation knowledge"""
        results = {
            "inferred_is_a": list(self.prolog.query("is_a(X, Y)")),
            "inferred_roles": list(self.prolog.query("chebi_has_role(X, Y)")),
            "custom_facts": list(self.prolog.query("user_relation(X, P, Y)"))
        }
        
        # Check consistency between ontology and annotations
        conflicts = list(self.prolog.query(
            "annotated_entity(X, _, _), chebi_class(Y), X == Y, "
            "not is_a(X, Y)"
        ))
        if conflicts:
            results["conflicts"] = conflicts
            
        return results

# Usage
if __name__ == "__main__":
    # Initialize with ChEBI ontology (download from:
    # https://www.ebi.ac.uk/chebi/downloadsForward.do)
    reasoner = ChebiReasoner("/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl")
    
    # Load user annotations
    reasoner.load_annotations("/home/matt/Proj/Hermeticav2/testing/AnnotationTesting/chebiRelAnnotations.json")
    
    # Perform integrated reasoning
    print(reasoner.full_reasoning())

PrologError: Caused by: 'chebi_has_role(X, Y)'. Returned: 'error(existence_error(procedure, /(chebi_has_role, 2)), context(/(pyrun, 2), _1106))'.

In [ ]:
from pyswip import Prolog
import json

class ChebiPipeline:
    def __init__(self, chebi_core_path):
        self.prolog = Prolog()
        self._load_core_ontology(chebi_core_path)
        
    def _load_core_ontology(self, path):
        """Load essential ChEBI taxonomy and properties"""
        # Load critical base classes
        self.prolog.assertz("chebi_class('CHEBI:24431')")  # chemical entity
        self.prolog.assertz("chebi_class('CHEBI:50860')")  # molecular entity
        
        # Load core relationships from file
        with open(path) as f:
            for line in f:
                self.prolog.assertz(line.strip())
                
    def load_formalized(self, formalized_output):
        """Load formulizer's logical statements"""
        for statement in formalized_output.split("\n"):
            if statement.endswith(".") and "(" in statement:
                try:
                    self.prolog.assertz(statement.strip())
                except Exception as e:
                    print(f"Failed to load: {statement}\nError: {str(e)}")
    
    def reason(self):
        """Perform ontology-compliant reasoning"""
        results = {
            "inconsistencies": [],
            "inferences": [],
            "matches": []
        }
        
        # Check hierarchy consistency
        results["inconsistencies"].extend(
            list(self.prolog.query("inconsistent"))
            
        # Infer new subclass relationships
        results["inferences"].extend(
            list(self.prolog.query("subclass(X, Y), not chebi_subclass(X, Y)"))
            
        # Find ontology-aligned annotations
        results["matches"].extend(
            list(self.prolog.query("annotated(X), chebi_class(X)"))
            
        return results

    def query(self, prolog_query):
        """Execute custom queries"""
        return list(self.prolog.query(prolog_query))

# Example Usage
if __name__ == "__main__":
    # 1. Formalization Stage (previous step)
    from formulizer import formulize_chebi_annotations
    
    with open("annotations.json") as f:
        annotations = json.load(f)
    formalized = formulize_chebi_annotations(annotations)
    
    # 2. Reasoning Stage
    pipeline = ChebiPipeline("chebi_core.pl")
    pipeline.load_formalized(formalized)
    
    # 3. Get results
    results = pipeline.reason()
    print("Inconsistencies:", results["inconsistencies"])
    print("New inferences:", results["inferences"])
    
    # Sample query: Find all solvents
    print("\nSolvents:", pipeline.query("has_role(X, 'CHEBI:46787')"))

# making a protoytype for ontological consistency with wikidata 

In [7]:
import json
from pyswip import Prolog
from SPARQLWrapper import SPARQLWrapper, JSON

WIKIDATA_SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"


def load_formulized_statements(input_file):
    """
    Loads formulized statements from JSON.
    """
    with open(input_file, "r") as f:
        return json.load(f)

def check_consistency(formulizations, wikidata_facts):
    """
    Uses Prolog to check if extracted statements are consistent with Wikidata.
    """
    prolog = Prolog()

    # Load Wikidata facts into Prolog
    for fact in wikidata_facts:
        subject, predicate, obj = map(escape_prolog_string, fact)  # Escape inputs
        prolog.assertz(f"wikidata_relation({subject}, {predicate}, {obj})")

    inconsistencies = []
    
    # Check if any extracted fact contradicts Wikidata
    for relation in formulizations["relationships"]:
        subject = escape_prolog_string(relation["subject"])
        predicate = escape_prolog_string(relation["predicate"])
        obj = escape_prolog_string(relation["object"])
        
        # Query Prolog to check if the fact exists in Wikidata
        prolog_query = f"wikidata_relation({subject}, {predicate}, {obj})"
        try:
            if not list(prolog.query(prolog_query)):
                inconsistencies.append({"subject": subject, "predicate": predicate, "object": obj, "status": "Inconsistent"})
        except Exception as e:
            print(f"Prolog Query Error: {e} for query: {prolog_query}")
    
    return inconsistencies

def escape_prolog_string(value):
    """
    Ensures safe formatting for Prolog terms.
    - Wraps string in single quotes if needed.
    - Escapes internal single quotes.
    """
    if isinstance(value, str):
        safe_value = value.replace("'", "\\'")  # Escape single quotes
        return f"'{safe_value}'" if " " in value or "'" in value else safe_value
    return value


if __name__ == "__main__":
    input_file = "/home/matt/Proj/Hermeticav2/src/reasoning/Formulizer/FormulizedStorage/formulizations.json"
    formulizations = load_formulized_statements(input_file)

    # Fetch existing knowledge from Wikidata
    wikidata_facts = fetch_wikidata_facts(formulizations["entities"], formulizations["relationships"])

    # Check consistency
    inconsistencies = check_consistency(formulizations, wikidata_facts)

    # Output results
    output_file = "consistency_report.json"
    with open(output_file, "w") as f:
        json.dump(inconsistencies, f, indent=4)
    
    print(f"Consistency check completed. Report saved to {output_file}")


Skipping invalid query: Coca__Cola, P170, in_1886
Skipping invalid query: Einstein_s_equation_E, P973, the_relationship_between_energy_and_mass
Consistency check completed. Report saved to consistency_report.json


In [2]:
from SPARQLWrapper import SPARQLWrapper, JSON

WIKIDATA_SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

def fetch_wikidata_facts(entities, relationships):
    """
    Queries Wikidata to retrieve known facts for consistency checking.
    """
    sparql = SPARQLWrapper(WIKIDATA_SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)
    
    # **Set a proper User-Agent header**
    sparql.addCustomHttpHeader("User-Agent", "HermeticaBot/1.0 (mailto:your-email@example.com)")
    
    wikidata_facts = set()
    
    for relation in relationships:
        subject, predicate, obj = relation["subject"], relation["predicate"], relation["object"]
        
        query = f"""
        SELECT ?objectLabel WHERE {{
            wd:{subject} wdt:{predicate} ?object.
            SERVICE wikibase:label {{ bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }}
        }}
        """
        sparql.setQuery(query)
        
        try:
            results = sparql.query().convert()
            for result in results["results"]["bindings"]:
                wikidata_facts.add((subject, predicate, result["objectLabel"]["value"]))
        
        except Exception as e:
            print(f"SPARQL Query Failed: {e}")
    
    return wikidata_facts


In [6]:
import re
from urllib.parse import quote
from SPARQLWrapper import SPARQLWrapper, JSON

WIKIDATA_SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

def clean_wikidata_id(value):
    """
    Cleans entity names for Wikidata queries:
    - Removes spaces and special characters if needed.
    - Ensures QIDs (e.g., 'Q42') and properties (e.g., 'P123') remain valid.
    - URL-encodes if necessary.
    """
    if value.startswith("Q") or value.startswith("P"):  # Valid Wikidata QID or property
        return value
    else:
        value = value.replace(" ", "_")  # Replace spaces with underscores
        value = re.sub(r"[^\w]", "", value)  # Remove special characters
        return quote(value)  # URL-encode for SPARQL

def fetch_wikidata_facts(entities, relationships):
    """
    Queries Wikidata to retrieve known facts for consistency checking.
    """
    sparql = SPARQLWrapper(WIKIDATA_SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)
    sparql.addCustomHttpHeader("User-Agent", "HermeticaBot/1.0 (mailto:your-email@example.com)")

    wikidata_facts = set()
    
    for relation in relationships:
        subject = clean_wikidata_id(relation["subject"])
        predicate = clean_wikidata_id(relation["predicate"])
        obj = clean_wikidata_id(relation["object"])
        
        if not subject.startswith("Q") or not predicate.startswith("P"):  # Ensure valid query structure
            print(f"Skipping invalid query: {subject}, {predicate}, {obj}")
            continue
        
        query = f"""
        SELECT ?objectLabel WHERE {{
            wd:{subject} wdt:{predicate} ?object.
            SERVICE wikibase:label {{ bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }}
        }}
        """
        
        sparql.setQuery(query)
        
        try:
            results = sparql.query().convert()
            for result in results["results"]["bindings"]:
                wikidata_facts.add((subject, predicate, result["objectLabel"]["value"]))
        except Exception as e:
            print(f"SPARQL Query Failed for {subject}, {predicate}: {e}")
    
    return wikidata_facts


In [ ]:
import json
import re
from urllib.parse import quote
from pyswip import Prolog
from SPARQLWrapper import SPARQLWrapper, JSON

WIKIDATA_SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

def clean_wikidata_id(value):
    """
    Cleans entity names for Wikidata queries:
    - Ensures QIDs (e.g., 'Q42') and properties (e.g., 'P123') remain valid.
    - Removes spaces and special characters if needed.
    - URL-encodes for SPARQL if necessary.
    """
    if value.startswith("Q") or value.startswith("P"):  # Valid Wikidata QID or property
        return value
    else:
        value = value.replace(" ", "_")  # Replace spaces with underscores
        value = re.sub(r"[^\w]", "", value)  # Remove special characters
        return quote(value)  # URL-encode for SPARQL

def clean_object_value(value):
    """
    Cleans object values by:
    - Removing unnecessary single quotes.
    - Fixing misplaced commas.
    - Ensuring proper Wikidata formatting.
    """
    value = value.strip("'")  # Remove surrounding single quotes
    value = value.replace(" , ", ", ")  # Fix misplaced commas
    return value

def fetch_wikidata_facts(entities, relationships):
    """
    Queries Wikidata to retrieve known facts for consistency checking.
    """
    sparql = SPARQLWrapper(WIKIDATA_SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)
    sparql.addCustomHttpHeader("User-Agent", "HermeticaBot/1.0 (mailto:mattthart@gmail.com)")

    wikidata_facts = set()
    
    for relation in relationships:
        subject = clean_wikidata_id(relation["subject"])
        predicate = clean_wikidata_id(relation["predicate"])
        obj = clean_object_value(relation["object"])
        
        if not subject.startswith("Q") or not predicate.startswith("P"):  # Ensure valid query structure
            print(f"Skipping invalid query: {subject}, {predicate}, {obj}")
            continue
        
        query = f"""
        SELECT ?objectLabel WHERE {{
            wd:{subject} wdt:{predicate} ?object.
            SERVICE wikibase:label {{ bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }}
        }}
        """
        
        sparql.setQuery(query)
        
        try:
            results = sparql.query().convert()
            for result in results["results"]["bindings"]:
                wikidata_facts.add((subject, predicate, result["objectLabel"]["value"]))
        except Exception as e:
            print(f"SPARQL Query Failed for {subject}, {predicate}: {e}")
    
    return wikidata_facts
def format_prolog_string(value):
    """
    Formats values correctly for Prolog:
    - Wraps in single quotes if needed.
    - Escapes internal single quotes.
    - Removes unwanted characters.
    """
    if isinstance(value, str):
        value = value.replace("'", "\\'")  # Escape single quotes for Prolog
        if " " in value or value.isalpha():  # Wrap in single quotes if it contains spaces
            return f"'{value}'"
    return value  # Return as-is for numbers or QIDs


def check_consistency(formulizations, wikidata_facts):
    """
    Uses Prolog to check if extracted statements are consistent with Wikidata.
    """
    prolog = Prolog()

    # Load Wikidata facts into Prolog
    for fact in wikidata_facts:
        subject, predicate, obj = map(format_prolog_string, fact)  # Apply formatting
        prolog.assertz(f"wikidata_relation({subject}, {predicate}, {obj})")

    inconsistencies = []
    
    # Check if any extracted fact contradicts Wikidata
    for relation in formulizations["relationships"]:
        subject = format_prolog_string(relation["subject"])
        predicate = format_prolog_string(relation["predicate"])
        obj = format_prolog_string(relation["object"])
        
        # Query Prolog to check if the fact exists in Wikidata
        prolog_query = f"wikidata_relation({subject}, {predicate}, {obj})"
        try:
            if not list(prolog.query(prolog_query)):
                inconsistencies.append({"subject": subject, "predicate": predicate, "object": obj, "status": "Inconsistent"})
        except Exception as e:
            print(f"Prolog Query Error: {e} for query: {prolog_query}")
    
    return inconsistencies


if __name__ == "__main__":
    input_file = "/home/matt/Proj/Hermeticav2/src/reasoning/Formulizer/FormulizedStorage/formulizations.json"
    output_file = "/home/matt/Proj/Hermeticav2/notebooks/prototyping/reasoningproto/consistency_report.json"

    # Load formulized statements
    with open(input_file, "r") as f:
        formulizations = json.load(f)

    # Fetch existing knowledge from Wikidata
    wikidata_facts = fetch_wikidata_facts(formulizations["entities"], formulizations["relationships"])

    # Check consistency using Prolog
    inconsistencies = check_consistency(formulizations, wikidata_facts)

    # Save results to a JSON file
    with open(output_file, "w") as f:
        json.dump(inconsistencies, f, indent=4)

    print(f"Consistency check completed. Report saved to {output_file}")


Skipping invalid query: Coca__Cola, P170, in 1886
Skipping invalid query: Einstein_s_equation_E, P973, the relationship between energy and mass


   Call: (1) pyrun("assertz((wikidata_relation(Q41255148, P276, 'Galleria Borghese'))).", _1644) ? 

# trying our fact inference form wikidat ontology 

In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON

# Define SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

# Query: Find subclass relationships
query = """
SELECT ?subclass ?subclassLabel WHERE {
  wd:Q937 wdt:P279 ?subclass.
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
"""
sparql.setQuery(query)

# Execute and print results
results = sparql.query().convert()
for result in results["results"]["bindings"]:
    print(result["subclass"]["value"], "-", result["subclassLabel"]["value"])


In [2]:
from SPARQLWrapper import SPARQLWrapper, JSON

def get_wikidata_facts(entity_qid):
    """Retrieve all known relationships for an entity from Wikidata"""
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    query = f"""
    SELECT ?predicate ?predicateLabel ?object ?objectLabel WHERE {{
      wd:{entity_qid} ?predicate ?object.
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }} LIMIT 50
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    
    results = sparql.query().convert()
    facts = []
    
    for result in results["results"]["bindings"]:
        predicate = result["predicateLabel"]["value"]
        obj = result["objectLabel"]["value"]
        facts.append((predicate, obj))
    
    return facts

def infer_facts(entity_qid):
    """Apply inference rules based on Wikidata relationships"""
    known_facts = get_wikidata_facts(entity_qid)
    inferred_facts = []

    for predicate, obj in known_facts:
        # Example: If Einstein is an instance of "Physicist", and Physicist is a subclass of Scientist,
        # we can infer Einstein is a Scientist.
        if predicate == "instance of":
            subclass_facts = get_wikidata_facts(obj)  # Look up the entity's subclass
            for sub_pred, sub_obj in subclass_facts:
                if sub_pred == "subclass of":
                    inferred_facts.append((f"{entity_qid} is also a", sub_obj))
    
    return known_facts, inferred_facts

# Test with Albert Einstein (Q937)
known, inferred = infer_facts("Q937")

print("\n✅ Known Facts:")
for fact in known:
    print(f"- {fact[0]}: {fact[1]}")

print("\n🚀 Inferred Facts:")
for fact in inferred:
    print(f"- {fact[0]} {fact[1]}")



✅ Known Facts:
- http://www.w3.org/2000/01/rdf-schema#label: अल्बर्ट आइंस्टीन
- http://www.w3.org/2000/01/rdf-schema#label: Albert Einstein
- http://www.w3.org/2000/01/rdf-schema#label: Albert Einstein
- http://www.w3.org/2000/01/rdf-schema#label: Albert Einstein
- http://www.w3.org/2000/01/rdf-schema#label: Albert Einstein
- http://www.w3.org/2000/01/rdf-schema#label: Albert Einstein
- http://www.w3.org/2000/01/rdf-schema#label: Ալբերտ Այնշտայն
- http://www.w3.org/2000/01/rdf-schema#label: Ալպերթ Այնշթայն
- http://www.w3.org/2000/01/rdf-schema#label: Albert Einstein
- http://www.w3.org/2000/01/rdf-schema#label: Albert Einstein
- http://www.w3.org/2000/01/rdf-schema#label: Albert Einstein
- http://www.w3.org/2000/01/rdf-schema#label: Albert Einstein
- http://www.w3.org/2000/01/rdf-schema#label: Albert Einstein
- http://www.w3.org/2000/01/rdf-schema#label: Albert Einstein
- http://www.w3.org/2000/01/rdf-schema#label: Albert Einstein
- http://www.w3.org/2000/01/rdf-schema#label: アルベルト・ア

In [3]:
from SPARQLWrapper import SPARQLWrapper, JSON

def get_wikidata_facts(entity_qid):
    """Retrieve all factual relationships for an entity from Wikidata"""
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    query = f"""
    SELECT ?predicate ?predicateLabel ?object ?objectLabel WHERE {{
      wd:{entity_qid} ?predicate ?object.
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
      FILTER (STRSTARTS(STR(?predicate), "http://www.wikidata.org/prop/direct/"))
    }} LIMIT 50
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    
    results = sparql.query().convert()
    facts = []
    
    for result in results["results"]["bindings"]:
        predicate = result["predicateLabel"]["value"]
        obj = result["objectLabel"]["value"]
        facts.append((predicate, obj))
    
    return facts

def infer_facts(entity_qid):
    """Apply inference rules based on Wikidata relationships"""
    known_facts = get_wikidata_facts(entity_qid)
    inferred_facts = []

    for predicate, obj in known_facts:
        # If the entity is an "instance of" something, check its subclass
        if predicate == "instance of":
            subclass_facts = get_wikidata_facts(obj)
            for sub_pred, sub_obj in subclass_facts:
                if sub_pred == "subclass of":
                    inferred_facts.append((f"{obj} is a subclass of", sub_obj))
                    inferred_facts.append((f"{entity_qid} is also a", sub_obj))
    
    return known_facts, inferred_facts

# Test with Albert Einstein (Q937)
known, inferred = infer_facts("Q937")

print("\n✅ Known Facts:")
for fact in known:
    print(f"- {fact[0]}: {fact[1]}")

print("\n🚀 Inferred Facts:")
for fact in inferred:
    print(f"- {fact[0]} {fact[1]}")



✅ Known Facts:
- http://www.wikidata.org/prop/direct/P18: http://commons.wikimedia.org/wiki/Special:FilePath/Albert%20Einstein%20Head.jpg
- http://www.wikidata.org/prop/direct/P19: Ulm
- http://www.wikidata.org/prop/direct/P20: Princeton
- http://www.wikidata.org/prop/direct/P21: male
- http://www.wikidata.org/prop/direct/P22: Hermann Einstein
- http://www.wikidata.org/prop/direct/P25: Pauline Koch
- http://www.wikidata.org/prop/direct/P26: Elsa Einstein
- http://www.wikidata.org/prop/direct/P26: Mileva Marić
- http://www.wikidata.org/prop/direct/P27: United States
- http://www.wikidata.org/prop/direct/P27: Switzerland
- http://www.wikidata.org/prop/direct/P27: Germany
- http://www.wikidata.org/prop/direct/P27: Weimar Republic
- http://www.wikidata.org/prop/direct/P27: German Empire
- http://www.wikidata.org/prop/direct/P27: statelessness
- http://www.wikidata.org/prop/direct/P27: Cisleithania
- http://www.wikidata.org/prop/direct/P31: human
- http://www.wikidata.org/prop/direct/P39: 

In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON

def get_wikidata_facts(entity_qid):
    """Retrieve all labeled relationships for an entity from Wikidata"""
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    query = f"""
    SELECT ?predicate ?predicateLabel ?object ?objectLabel WHERE {{
      wd:{entity_qid} ?predicate ?object.
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
      FILTER (STRSTARTS(STR(?predicate), "http://www.wikidata.org/prop/direct/"))
    }} LIMIT 50
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    
    results = sparql.query().convert()
    facts = []
    
    for result in results["results"]["bindings"]:
        predicate = result["predicateLabel"]["value"]
        obj = result["objectLabel"]["value"]
        facts.append((predicate, obj))
    
    return facts

def infer_facts(entity_qid):
    """Apply reasoning on known facts"""
    known_facts = get_wikidata_facts(entity_qid)
    inferred_facts = []

    for predicate, obj in known_facts:
        # If an entity is an "instance of" something, check its subclass
        if predicate == "instance of":
            subclass_facts = get_wikidata_facts(obj)
            for sub_pred, sub_obj in subclass_facts:
                if sub_pred == "subclass of":
                    inferred_facts.append((f"{obj} is a subclass of", sub_obj))
                    inferred_facts.append((f"{entity_qid} is also a", sub_obj))
    
    return known_facts, inferred_facts

# Test with Albert Einstein (Q937)
known, inferred = infer_facts("Q937")

print("\n✅ Known Facts:")
for fact in known:
    print(f"- {fact[0]}: {fact[1]}")

print("\n🚀 Inferred Facts:")
for fact in inferred:
    print(f"- {fact[0]} {fact[1]}")



✅ Known Facts:
- http://www.wikidata.org/prop/direct/P18: http://commons.wikimedia.org/wiki/Special:FilePath/Albert%20Einstein%20Head.jpg
- http://www.wikidata.org/prop/direct/P19: Ulm
- http://www.wikidata.org/prop/direct/P20: Princeton
- http://www.wikidata.org/prop/direct/P21: male
- http://www.wikidata.org/prop/direct/P22: Hermann Einstein
- http://www.wikidata.org/prop/direct/P25: Pauline Koch
- http://www.wikidata.org/prop/direct/P26: Elsa Einstein
- http://www.wikidata.org/prop/direct/P26: Mileva Marić
- http://www.wikidata.org/prop/direct/P27: United States
- http://www.wikidata.org/prop/direct/P27: Switzerland
- http://www.wikidata.org/prop/direct/P27: Germany
- http://www.wikidata.org/prop/direct/P27: Weimar Republic
- http://www.wikidata.org/prop/direct/P27: German Empire
- http://www.wikidata.org/prop/direct/P27: statelessness
- http://www.wikidata.org/prop/direct/P27: Cisleithania
- http://www.wikidata.org/prop/direct/P31: human
- http://www.wikidata.org/prop/direct/P39: 

In [14]:
from SPARQLWrapper import SPARQLWrapper, JSON

def query_wikidata(query):
    """Run a SPARQL query and return results."""
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    return results["results"]["bindings"]

def get_facts(entity_qid):
    """Retrieve all relationships (known facts) for an entity in English."""
    query = f"""
    SELECT ?propertyLabel ?valueLabel WHERE {{
      wd:{entity_qid} ?property ?value.
      ?property rdfs:label ?propertyLabel.
      ?value rdfs:label ?valueLabel.
      FILTER(LANG(?propertyLabel) = "en" && LANG(?valueLabel) = "en").
    }} LIMIT 200
    """
    results = query_wikidata(query)
    return [(res["propertyLabel"]["value"], res["valueLabel"]["value"]) for res in results]

def get_taxonomy(entity_qid):
    """Retrieve subclass relationships (parent classes)."""
    query = f"""
    SELECT ?superclassLabel WHERE {{
      wd:{entity_qid} wdt:P279 ?superclass.
      ?superclass rdfs:label ?superclassLabel.
      FILTER(LANG(?superclassLabel) = "en").
    }} LIMIT 20
    """
    results = query_wikidata(query)
    return [res["superclassLabel"]["value"] for res in results]

def get_all_info(entity_qid):
    """Retrieve both known facts and taxonomy for an entity."""
    known_facts = get_facts(entity_qid)
    taxonomy = get_taxonomy(entity_qid)

    print("\n✅ Known Facts:")
    if known_facts:
        for fact in known_facts:
            print(f"- {fact[0]}: {fact[1]}")
    else:
        print("No known facts found.")

    print("\n🚀 Taxonomy (Parent Classes):")
    if taxonomy:
        for parent in taxonomy:
            print(f"- {entity_qid} is a subclass of {parent}")
    else:
        print("No taxonomy information found.")

# Example: Fetch data for Albert Einstein (Q937)
get_all_info("Q937")



✅ Known Facts:
No known facts found.

🚀 Taxonomy (Parent Classes):
No taxonomy information found.


In [2]:
VERBALIZATION_TEMPLATES = {
    'is_a': lambda s, o: f"{s} is a type of {o}",
    'has_part': lambda s, o: f"{s} contains {o}",
    'has_role': lambda s, o: f"{s} functions as {o}",
    'reacts_with': lambda s, o: f"{s} reacts with {o}",
    'subclass': lambda s, o: f"{s} is a specialized form of {o}",
    'default': lambda p, s, o: f"{s} {p.replace('_', ' ')} {o}"
}

In [10]:

VERBALIZATION_TEMPLATES = {
    'is_a': lambda s, o: f"{s} is a type of {o}",
    'has_part': lambda s, o: f"{s} contains {o}",
    'has_role': lambda s, o: f"{s} functions as {o}",
    'reacts_with': lambda s, o: f"{s} reacts with {o}",
    'subclass': lambda s, o: f"{s} is a specialized form of {o}",
    'default': lambda p, s, o: f"{s} {p.replace('_', ' ')} {o}"
}


class Verbalizer:
    def __init__(self, chebi_labels):
        self.label_map = chebi_labels  # {'CHEBI:15377': 'water', ...}
        
    def _get_label(self, entity_id):
        return self.label_map.get(entity_id, entity_id)




def verbalize_statement(self, statement):
    if isinstance(statement, dict):  # Prolog query result
        pred = statement['Predicate']
        subj = self._get_label(statement['Subject'])
        obj = self._get_label(statement['Object'])
    else:  # String statement
        parts = statement.strip(' .').split('(')
        pred = parts[0]
        args = parts[1].split(', ')
        subj = self._get_label(args[0].strip("'"))
        obj = self._get_label(args[1].strip("'"))

    template = VERBALIZATION_TEMPLATES.get(pred, VERBALIZATION_TEMPLATES['default'])
    return template(subj, obj)



def group_statements(self, statements):
    entity_groups = defaultdict(list)
    for stmt in statements:
        entity = stmt.split('(')[1].split(',')[0].strip("'")
        entity_groups[entity].append(stmt)
    return entity_groups


def generate_paragraph(self, grouped_statements):
    paragraphs = []
    for entity, stmts in grouped_statements.items():
        entity_name = self._get_label(entity)
        paragraph = [f"{entity_name} is a chemical entity with these properties:"]
        
        for stmt in stmts:
            verbalized = self.verbalize_statement(stmt)
            paragraph.append(f"- {verbalized.capitalize()}.")
            
        paragraphs.append(" ".join(paragraph))
    
    return "\n\n".join(paragraphs)



def mark_inferences(self, statements, inferred):
    return [
        f"{self.verbalize_statement(s)} [Inferred]" if s in inferred 
        else self.verbalize_statement(s)
        for s in statements
    ]


class ChebiVerbalizer:
    def __init__(self, chebi_labels):
        self.verbalizer = Verbalizer(chebi_labels)
        
    def convert_to_natural_language(self, original_statements, inferred_statements):
        # Combine and mark inferences
        all_statements = original_statements + inferred_statements
        marked = self.verbalizer.mark_inferences(all_statements, inferred_statements)
        
        # Group and structure
        grouped = self.verbalizer.group_statements(marked)
        
        # Generate final text
        return self.verbalizer.generate_paragraph(grouped)



chebi_labels= "is_a('CHEBI:16234', 'CHEBI:138505') has_role('CHEBI:138505', 'CHEBI:23835') subclass('CHEBI:16234', 'CHEBI:138505')"   

# Usage Example
verbalizer = ChebiVerbalizer(chebi_labels)
natural_language = verbalizer.convert_to_natural_language(
    original_formalized, 
    new_inferences
)

NameError: name 'original_formalized' is not defined

In [12]:
from collections import defaultdict

VERBALIZATION_TEMPLATES = {
    'is_a': lambda s, o: f"{s} is a type of {o}",
    'has_role': lambda s, o: f"{s} functions as {o}",
    'subclass': lambda s, o: f"{s} is a specialized form of {o}",
    'default': lambda p, s, o: f"{s} {p.replace('_', ' ')} {o}"
}

class Verbalizer:
    def __init__(self, chebi_labels):
        self.label_map = chebi_labels  # Should be a dictionary
        
    def _get_label(self, entity_id):
        return self.label_map.get(entity_id.strip("'"), entity_id)
    
    def verbalize_statement(self, statement):
        if isinstance(statement, dict):
            pred = statement['Predicate']
            subj = self._get_label(statement['Subject'])
            obj = self._get_label(statement['Object'])
        else:
            parts = statement.strip(' .').split('(')
            pred = parts[0]
            args = parts[1].rstrip(')').split(', ')
            subj = self._get_label(args[0].strip("'"))
            obj = self._get_label(args[1].strip("'"))
        
        template = VERBALIZATION_TEMPLATES.get(pred, VERBALIZATION_TEMPLATES['default'])
        return template(subj, obj) if pred != 'default' else template(pred, subj, obj)
    
    def group_statements(self, statements):
        entity_groups = defaultdict(list)
        for stmt in statements:
            if '(' in stmt and ',' in stmt:
                entity = stmt.split('(')[1].split(',')[0].strip("'")
                entity_groups[entity].append(stmt)
        return entity_groups
    
    def generate_paragraph(self, grouped_statements):
        paragraphs = []
        for entity, stmts in grouped_statements.items():
            entity_name = self._get_label(entity)
            paragraph = [f"{entity_name} has these properties:"]
            for stmt in stmts:
                verbalized = self.verbalize_statement(stmt)
                paragraph.append(f"- {verbalized.capitalize()}.")
            paragraphs.append("\n".join(paragraph))
        return "\n\n".join(paragraphs)
    
    def mark_inferences(self, statements, inferred):
        return [
            f"{self.verbalize_statement(s)} [Inferred]" if s in inferred 
            else self.verbalize_statement(s)
            for s in statements
        ]

class ChebiVerbalizer:
    def __init__(self, chebi_labels):
        self.verbalizer = Verbalizer(chebi_labels)
        
    def convert_to_natural_language(self, original_statements, inferred_statements):
        all_statements = original_statements + inferred_statements
        marked = self.verbalizer.mark_inferences(all_statements, inferred_statements)
        grouped = self.verbalizer.group_statements(marked)
        return self.verbalizer.generate_paragraph(grouped)

# Sample data
chebi_labels = {
    'CHEBI:16234': 'ethanol',
    'CHEBI:138505': 'alcohol',
    'CHEBI:23835': 'solvent'
}

original_formalized = [
    "is_a('CHEBI:16234', 'CHEBI:138505')",
    "has_role('CHEBI:138505', 'CHEBI:23835')"
]

new_inferences = [
    "subclass('CHEBI:16234', 'CHEBI:138505')"
]

# Usage Example
verbalizer = ChebiVerbalizer(chebi_labels)
natural_language = verbalizer.convert_to_natural_language(
    original_formalized, 
    new_inferences
)

print(natural_language)